In [1]:
from build123d import (
    Align,
    Axis,
    Box,
    BuildPart,
    BasePartObject,
    BuildLine,
    BuildSketch,
    Circle,
    FilletPolyline,
    Line,
    Locations,
    MM,
    Mode,
    Polyline,
    Plane,
    RectangleRounded,
    RotationLike,
    add,
    extrude,
    export_stl,
    make_face,
    mirror,
    revolve,
)
from ocp_vscode import set_port, show
from gridfinity_build123d import BaseEqual

# Set the desired port number
set_port(3939)

BASE_LENGTH = 6 # Units
BASE_WIDTH = 4 # Units
BASE_CORNER_RADIUS = 7.5 / 2 * MM # mm
HEIGHT = 63 * MM # mm

CHOP_LENGTH = 220 * MM # mm
CHOP_WIDTH  = 160 * MM # mm
CHOP_CORNER_RADIUS = 35 * MM # mm
CHOP_HEIGHT = (HEIGHT - 7) * MM # mm

SIDE_DOUBLE_LENGTH = 75 * MM # mm from outside edge of bin wall to edge of cutout
SIDE_HALF_LENGTH = (BASE_LENGTH * 42 * MM) / 2 # mm from centerline to outside edge of bin wall
CUTOUT_LENGTH = SIDE_HALF_LENGTH - SIDE_DOUBLE_LENGTH # mm from centerline to edge of cutout 
CUTOUT_RADIUS = 12.5 * MM # mm radius of side cutout arc
CUTOUT_ARC = CUTOUT_LENGTH + CUTOUT_RADIUS + 0.1 * MM # mm from centerline to edge of cutout arc

In [2]:
BIN_BASE = BaseEqual(
    grid_x=BASE_WIDTH, 
    grid_y=BASE_LENGTH,
)

show(BIN_BASE)


+


In [3]:
half_length=SIDE_HALF_LENGTH
double_length=SIDE_DOUBLE_LENGTH 
height=CHOP_HEIGHT 

with BuildLine() as chop_outline:
    FilletPolyline(
        (0, 0),
        (CUTOUT_LENGTH, 0),
        (CUTOUT_LENGTH, CHOP_HEIGHT),
        (CUTOUT_ARC, CHOP_HEIGHT),
        radius=CUTOUT_RADIUS,
    )
    # Line((0, CHOP_HEIGHT * MM))
    Line(
        (0, CHOP_HEIGHT),
        (CUTOUT_ARC, CHOP_HEIGHT),
    )
    mirror(about=Plane.YZ)
    
show(chop_outline)

In [4]:
with BuildSketch() as chop_profile:
    add(chop_outline)
    make_face()
    
show(chop_profile)


-

+


In [5]:
with BuildPart() as test_block:
    add(
        Box(
            height=CHOP_HEIGHT, 
            width=BASE_WIDTH * 42 * MM,
            length=BASE_LENGTH * 42 * MM,
        )
    )
    with BuildSketch([test_block.faces().sort_by(Axis.Z)[0], test_block.faces().sort_by(Axis.Z)[-1]]):
        Circle(
            radius=CHOP_CORNER_RADIUS,
            align=Align.CENTER,
        )

    # Extrude the cutout from long side of bin
    extrude(amount=-15, mode=Mode.SUBTRACT)
    with BuildSketch(
        [
            test_block.faces().sort_by(Axis.X)[0], test_block.faces().sort_by(Axis.X)[-1],
            test_block.faces().sort_by(Axis.Y)[0], test_block.faces().sort_by(Axis.Y)[-1],
            test_block.faces().sort_by(Axis.Z)[0], test_block.faces().sort_by(Axis.Z)[-1],
        ]
    ):
        RectangleRounded(
            height=CHOP_CORNER_RADIUS,
            width=CHOP_CORNER_RADIUS,
            radius=5 * MM,
            align=Align.CENTER,
        )
    extrude(amount=-30, mode=Mode.SUBTRACT)

show(test_block)

+


In [7]:
class TestBlock:
    def __init__(self):

        with BuildPart() as test_block:
            self.block = test_block

            add(
                Box(
                    height=CHOP_HEIGHT, 
                    width=BASE_WIDTH * 42 * MM,
                    length=BASE_LENGTH * 42 * MM,
                )
            )
            with BuildSketch([test_block.faces().sort_by(Axis.Z)[0], test_block.faces().sort_by(Axis.Z)[-1]]):
                Circle(
                    radius=CHOP_CORNER_RADIUS,
                    align=Align.CENTER,
                )

            # Extrude the cutout from long side of bin
            extrude(amount=-15, mode=Mode.SUBTRACT)
            with BuildSketch(
                [
                    test_block.faces().sort_by(Axis.X)[0], test_block.faces().sort_by(Axis.X)[-1],
                    test_block.faces().sort_by(Axis.Y)[0], test_block.faces().sort_by(Axis.Y)[-1],
                    test_block.faces().sort_by(Axis.Z)[0], test_block.faces().sort_by(Axis.Z)[-1],
                ]
            ):
                RectangleRounded(
                    height=CHOP_CORNER_RADIUS,
                    width=CHOP_CORNER_RADIUS,
                    radius=5 * MM,
                    align=Align.CENTER,
                )
            extrude(amount=-30, mode=Mode.SUBTRACT)

test_block_instance = TestBlock()
show(test_block_instance.block)

c


In [8]:
with BuildPart() as bin:
    # Add the base
    add(BIN_BASE)
    # Add a sketch on top for the chop compartment
    with BuildSketch(bin.faces().sort_by(Axis.Z)[-1]) as chop_sketch:

        RectangleRounded(
            height=BASE_LENGTH * 42 * MM,
            width=BASE_WIDTH * 42 * MM,
            radius=BASE_CORNER_RADIUS,
            align=(Align.CENTER, Align.CENTER)
        )
        RectangleRounded(
            height=CHOP_LENGTH,
            width=CHOP_WIDTH,
            radius=CHOP_CORNER_RADIUS,
            mode=Mode.SUBTRACT,
            align=(Align.CENTER, Align.CENTER)
        )
    # Extrude the bin to the specified height
    extrude(to_extrude=chop_sketch.face(), amount=CHOP_HEIGHT)
    # Add the cutout on the side for the chopping boards
    # with BuildSketch([bin.faces().sort_by(Axis.X)[0],bin.faces().sort_by(Axis.X)[-1]]):
    #     with BuildLine():
    #         add(chop_outline)
    #         make_face()

    # Extrude the cutout from long side of bin
    # extrude(amount=-2, mode=Mode.SUBTRACT)
    # extrude(amount=BASE_WIDTH * 42 * -1 * MM, mode=Mode.SUBTRACT)

show(bin)


+


In [ ]:
class ChopBin(BasePartObject):
    """Gridfinity Bin object with quirky compartment for storing IKEA chopping boards."""

    def __init__(
        self,
        height: float = 0,
        height_in_units: int = 0,
        rotation: RotationLike = (0, 0, 0),
        align: Align | tuple[Align, Align, Align] | None = None,
        mode: Mode = Mode.ADD,
    ):
        """Construct a custom bin object.

        Args:
            base (Part): Base object on which the bin is constructed.
            height (float, optional): Height of the bin in mm. Can't be used when height_in_units is
                defined.Defaults to 0.
            height_in_units (int, optional): Heigth defined by gridfinity units. Can't be used when
                height is defined. Defaults to 0.
            compartment (Compartment | None): Custom compartment of the bin, Defaults to None.
            rotation (RotationLike, optional): angles to rotate about axes. Defaults to (0, 0, 0).
            align (Union[Align, tuple[Align, Align, Align]], optional): align min, center, or max
            of object. Defaults to None.
            mode (Mode, optional): combination mode. Defaults to Mode.ADD.
        """
        if height and height_in_units:
            msg = "height or height_in_units can be defined, not both"
            raise ValueError(msg)
        if height_in_units:
            bin_height = height_in_units * 7
        else:
            bin_height = height

        with BuildPart() as bin:
            # Add the base
            add(
                BaseEqual(
                    grid_x=BASE_WIDTH, 
                    grid_y=BASE_LENGTH,
                    rotation=rotation,
                    align=align,
                    mode=mode
                )
            )
            # Add a sketch on top for the chop compartment
            with BuildSketch(bin.faces().sort_by(Axis.Z)[-1]) as chop_sketch:

                RectangleRounded(
                    height=BASE_LENGTH * 42 * MM, 
                    width=BASE_WIDTH * 42 * MM,
                    radius=BASE_CORNER_RADIUS * MM,
                    align=(Align.CENTER, Align.CENTER)
                )
                RectangleRounded(
                    height=CHOP_LENGTH * MM, 
                    width=CHOP_WIDTH * MM,
                    radius=CHOP_CORNER_RADIUS * MM,
                    mode=Mode.SUBTRACT,
                    align=(Align.CENTER, Align.CENTER)
                )
            # Extrude the bin to the specified height
            extrude(to_extrude=chop_sketch.face(), amount=bin_height)
            # Add the cutout on the side for the chopping boards
            # side_face = bin.faces().sort_by(Axis.X)[0]
            # with BuildSketch(side_face):
            #     with BuildLine():
            #         FilletPolyline(
            #             (SIDE_HALF_LENGTH * MM, CHOP_HEIGHT * MM),
            #             (SIDE_HALF_LENGTH * MM - SIDE_DOUBLE_LENGTH * MM, CHOP_HEIGHT * MM),
            #             (SIDE_HALF_LENGTH * MM - SIDE_DOUBLE_LENGTH * MM, 0),
            #             (0, 0),
            #             radius=SIDE_RADIUS * MM,
            #         )
            #         Polyline(
            #             (0, 0),
            #             (0, CHOP_HEIGHT * MM),
            #             (SIDE_HALF_LENGTH * MM, CHOP_HEIGHT * MM),
            #         )

            #     # mirror(about=Plane.YZ)
            # # Extrude the cutout from long side of bin
            # extrude(amount=-2, mode=Mode.SUBTRACT)
            # # extrude(amount=BASE_WIDTH * 42 * -1 * MM, mode=Mode.SUBTRACT)

        super().__init__(bin.part, rotation, align, mode)

chop_block = ChopBin(
    height=CHOP_HEIGHT
)
# show(chop_block)
show(chop_block)
# export_stl(chop_block, "chop_block.stl")

c
